# Developed Market v7 — A v E Offset + Two-Part Model + IBNR Development

Key changes from v6:
1. **A v E offset**: `log(expected_claim)` = `log(exposure x written_premium x LR_2021)` instead of `log(exposure)`
2. **Year removed** from features (was learning development inflation, not trend)
3. **written_premium removed** from features (now in the offset)
4. **Two-part model**: Frequency (binary XGBoost) + Severity (Gamma XGBoost) alongside the Tweedie model
5. **External LDFs for BI/subsidence only** (short-tail perils use data-driven development)
6. Data-driven development from chain-ladder + paid/incurred (in data prep)

In [0]:
# Step 1: Load v7 proc files + apply external LDFs for BI/subsidence ONLY
import pandas as pd
import numpy as np
import os, json

%pip install xgboost statsmodels openpyxl -q

BASE = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026'
DATA_DIR = f'{BASE}/data/'
V7_DIR = f'{BASE}/files_v7/'
SUB_DIR = f'{BASE}/submissions_v7/'
os.makedirs(SUB_DIR, exist_ok=True)

# Load LR_2021
with open(os.path.join(V7_DIR, 'lr_2021.json')) as f:
    LR_2021 = {k: float(v) for k, v in json.load(f).items()}
print(f'LR_2021: {LR_2021}')

# === EXTERNAL LDFs for BI and subsidence only ===
RAW_LDFS = {
    'bodily_injury': {12: (1.85, 2.40), 24: (1.40, 1.65), 36: (1.20, 1.35), 48: (1.10, 1.18), 60: (1.05, 1.08), 'tail': (1.02, 1.05)},
    'subsidence': {12: (1.50, 1.80), 24: (1.15, 1.25), 36: (1.05, 1.10), 48: (1.01, 1.03), 60: (1.00, 1.00), 'tail': (1.00, 1.00)},
}
LDF_MID = {p: {a: (r[0]+r[1])/2 for a, r in f.items()} for p, f in RAW_LDFS.items()}
SNAPSHOT = pd.Timestamp('2024-12-31')
loss_years = [2021, 2022, 2023, 2024]
dev_ages = {yr: (SNAPSHOT - pd.Timestamp(f'{yr}-07-01')).days / 30.44 for yr in loss_years}

def interp_ldf(peril, age):
    t = LDF_MID[peril]; ages = [12, 24, 36, 48, 60]
    if age <= 12: return t[12]
    elif age >= 60: return t[60] * t['tail']
    for i in range(len(ages)-1):
        if ages[i] <= age <= ages[i+1]:
            frac = (age - ages[i]) / (ages[i+1] - ages[i])
            return t[ages[i]] + frac * (t[ages[i+1]] - t[ages[i]])
    return t[60]

YEAR_LDFS = {yr: {'bodily_injury': interp_ldf('bodily_injury', dev_ages[yr]),
                  'subsidence': interp_ldf('subsidence', dev_ages[yr])} for yr in loss_years}

# Data-driven LDFs from Step 5 of data prep (for comparison)
# These are already applied to ultimate_claims in v7 proc files
# External LDFs only ADD additional uplift for long-tail perils (BI, subsidence)
# The data-driven LDF for 2024 is ~1.25; external LDF for BI is ~2.2, subsidence ~1.6
# We apply the RATIO of external to data-driven as additional scaling
# But since the data-driven LDF is aggregate (all perils), and external is peril-specific,
# we compute the policy-level scaling as before but ONLY for BI/subsidence claims

# For short-tail perils, the data-driven development is sufficient (no external LDF adjustment)
# For BI/subsidence, the external LDF provides additional tail development
# We need to compute: external_factor / data_driven_factor for BI/subsidence claims

# The data-driven combined LDFs by year (from data prep output):
DATA_DRIVEN_LDF = {2020: 1.0052, 2021: 1.0055, 2022: 1.0105, 2023: 1.0259, 2024: 1.2475}

# Load claims to compute BI/subsidence scaling
claims = pd.read_parquet(f'{BASE}/data/claims.parquet')
non_id = [c for c in claims.columns if c != 'claim_id']
claims = claims.drop_duplicates(subset=non_id, keep='first').reset_index(drop=True)
claims['loss_year'] = pd.to_datetime(claims['loss_date']).dt.year
claims['incurred_net'] = claims['incurred_to_date'] - claims['recoveries']
claims = claims[claims['status'] != 'repudiated'].copy()
claims = claims[claims['loss_year'].isin([2021, 2022, 2023, 2024])].copy()

# Compute external LDF for BI and subsidence only
claims['external_ldf'] = 1.0
for idx, row in claims.iterrows():
    p = str(row['peril']).lower()
    yr = row['loss_year']
    if p == 'bodily_injury':
        claims.at[idx, 'external_ldf'] = YEAR_LDFS.get(yr, {}).get('bodily_injury', 1.0)
    elif p == 'subsidence':
        claims.at[idx, 'external_ldf'] = YEAR_LDFS.get(yr, {}).get('subsidence', 1.0)

# Data-driven LDF per claim
claims['data_driven_ldf'] = claims['loss_year'].map(DATA_DRIVEN_LDF).fillna(1.0)

# Additional scaling = external / data_driven (only for BI/subsidence, =1 for others)
claims['additional_scaling'] = np.where(
    claims['external_ldf'] > 1.0,
    claims['external_ldf'] / claims['data_driven_ldf'],
    1.0
)

# Policy-level scaling (weighted average of additional_scaling by incurred_net)
bi_subs_claims = claims[claims['additional_scaling'] > 1.0]
print(f'BI/subsidence claims: {len(bi_subs_claims)}')
if len(bi_subs_claims) > 0:
    policy_scaling = bi_subs_claims.groupby(['line', 'policy_id']).apply(
        lambda g: np.average(g['additional_scaling'], weights=g['incurred_net']) if g['incurred_net'].sum() > 0 else 1.0
    ).reset_index(name='scaling_factor')
    print(f'Policies with BI/subsidence scaling: {len(policy_scaling)}')
    print(f'Mean scaling: {policy_scaling["scaling_factor"].mean():.4f}')
else:
    policy_scaling = pd.DataFrame(columns=['line', 'policy_id', 'scaling_factor'])

# === UPDATE V7 PROC FILES with external LDF scaling for BI/subsidence ===
train_v7 = {}
for line in ['motor', 'buildings', 'contents']:
    train_v7[line] = pd.read_parquet(os.path.join(V7_DIR, f'proc_{line}_v7.parquet'))
    old_mean = train_v7[line]['ultimate_claims'].mean()
    ls = policy_scaling[policy_scaling['line'] == line][['policy_id', 'scaling_factor']]
    if len(ls) > 0:
        train_v7[line] = train_v7[line].merge(ls, on='policy_id', how='left')
        train_v7[line]['scaling_factor'] = train_v7[line]['scaling_factor'].fillna(1.0)
        train_v7[line]['ultimate_claims'] = train_v7[line]['ultimate_claims'] * train_v7[line]['scaling_factor']
        train_v7[line].drop('scaling_factor', axis=1, inplace=True)
    new_mean = train_v7[line]['ultimate_claims'].mean()
    print(f'{line}: old mean={old_mean:.4f}, new mean={new_mean:.4f}')
print('\nStep 1 complete.')

In [0]:
# Step 2: Test prep + A v E XGBoost (offset = log(expected_claim), Year + written_premium removed)
import xgboost as xgb

# Load test files + enrichment
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Clean test files (same as v6)
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    df['Year'] = ps.dt.year
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    if 'compulsory_excess' in df.columns:
        df.drop('compulsory_excess', axis=1, inplace=True)

# Sum insured fix
for line in ['buildings', 'contents']:
    df = test[line]
    if 'sum_insured' in df.columns:
        threshold = 10_000_000 if line == 'buildings' else 1_000_000
        flagged = df['sum_insured'] >= threshold
        if flagged.sum() > 0:
            df.loc[flagged, 'sum_insured'] = df.loc[flagged, 'sum_insured'] / 1000
            print(f'test_{line}: {flagged.sum()} sum_insured values fixed')

# Construction type harmonization
CONSTRUCTION_MAP_DEV = {'BRICK': 'BRICK', 'STONE': 'STONE', 'W': 'TIMBER', 'T': 'TIMBER',
                        'NON_STANDARD': 'NON_STANDARD', 'TIMBER': 'TIMBER'}
test['buildings']['construction_type'] = test['buildings']['construction_type'].astype(str).str.strip().str.upper().replace(CONSTRUCTION_MAP_DEV)

# Zero-pad motor customer_id
test['motor']['customer_id'] = test['motor']['customer_id'].str.zfill(8)
customers['customer_id'] = customers['customer_id'].astype(str).str.zfill(8)

# Add age_band
def to_age_band(age):
    if pd.isna(age): return None
    if age <= 24: return '17-24'
    elif age <= 34: return '25-34'
    elif age <= 49: return '35-49'
    elif age <= 64: return '50-64'
    else: return '65+'
test['motor']['driver_age_band'] = test['motor']['driver_age'].apply(to_age_band)
for line in ['buildings', 'contents']:
    test[line]['policyholder_age_band'] = test[line]['policyholder_age'].apply(to_age_band)

# Join enrichment
customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    # Create expected_claim for test data
    df['expected_claim'] = df['exposure'] * df['written_premium'] * LR_2021[line]
    test[line] = df
    print(f'test_{line}: {df.shape}, expected_claim mean={df["expected_claim"].mean():.2f}')

# Feature selection (v6 Features_v3.xlsx) - NOW EXCLUDING Year and written_premium
FEATURES_PATH = f'{BASE}/notebooks_v4/Features_v3.xlsx'
sheet_map = {'motor': 'Motor', 'buildings': 'Building', 'contents': 'Contents'}
feature_selection = {}
EXCLUDE_FROM_FEATS = {'Year', 'written_premium', 'expected_claim', 'ae_ratio',
                      'ultimate_claims', 'row_id', 'policy_id', 'customer_id', 'address_id',
                      'period_start', 'period_end', 'claim_count', 'reported_incurred',
                      'open_claim_count'}
for line, sheet in sheet_map.items():
    df = pd.read_excel(FEATURES_PATH, sheet_name=sheet)
    df.iloc[:, 0] = df.iloc[:, 0].astype(str).str.strip()
    feats = df[df.iloc[:, 1] == 1].iloc[:, 0].tolist()
    feats = [f for f in feats if f not in EXCLUDE_FROM_FEATS]
    feature_selection[line] = feats
    print(f'{line}: {len(feats)} features (Year + written_premium excluded)')

# Tweedie D2
def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

XGB_PARAMS = {
    'objective': 'reg:tweedie', 'tweedie_variance_power': 1.5,
    'max_depth': 6, 'eta': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.5,
    'min_child_weight': 10, 'reg_alpha': 0.5, 'reg_lambda': 2.0,
    'seed': 42, 'n_jobs': -1, 'verbosity': 0,
}
NEW_TREND = {'motor': 1.048, 'buildings': 1.090, 'contents': 1.049}

xgb_models = {}
xgb_val_d2 = {}
xgb_val_preds = {}
xgb_test_preds = {}
xgb_importance = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}\n=== {line.upper()} A v E XGBoost v7 ===\n{"="*60}')
    train_df = train_v7[line]
    test_df = test[line]
    feats = [f for f in feature_selection[line] if f in train_df.columns and f in test_df.columns]
    print(f'  Features: {len(feats)}')
    
    X_all = train_df[feats].copy()
    X_test = test_df[feats].copy()
    for col in feats:
        if X_all[col].dtype == 'object' or X_all[col].dtype == 'bool':
            cats = pd.Categorical(X_all[col])
            X_all[col] = cats.codes.astype('float64')
            X_test[col] = pd.Categorical(X_test[col], categories=cats.categories).codes.astype('float64')
    
    y_all = train_df['ultimate_claims'].values
    year = train_df['Year'].values
    train_mask = year <= 2023
    val_mask = year == 2024
    
    # A v E OFFSET: log(expected_claim) instead of log(exposure)
    offset_all = np.log(train_df['expected_claim'].values + 1e-10)
    offset_test = np.log(test_df['expected_claim'].values + 1e-10)
    weight_all = train_df['exposure'].values  # still weight by exposure
    
    dtrain = xgb.DMatrix(X_all[train_mask], label=y_all[train_mask],
                         base_margin=offset_all[train_mask], weight=weight_all[train_mask])
    dval = xgb.DMatrix(X_all[val_mask], label=y_all[val_mask],
                       base_margin=offset_all[val_mask], weight=weight_all[val_mask])
    dtest = xgb.DMatrix(X_test, base_margin=offset_test)
    
    model = xgb.train(XGB_PARAMS, dtrain, num_boost_round=300,
                      evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    xgb_models[line] = model
    val_pred = np.maximum(model.predict(dval), 0)
    d2 = tweedie_d2(y_all[val_mask], val_pred)
    xgb_val_d2[line] = d2
    xgb_val_preds[line] = val_pred
    print(f'  Val D2: {d2:.4f}')
    print(f'  Val pred mean: {val_pred.mean():.4f} (actual: {y_all[val_mask].mean():.4f})')
    print(f'  Val A/E: {val_pred.mean() / train_df.loc[val_mask, "expected_claim"].mean():.4f}')
    scores = model.get_score(importance_type='gain')
    imp = pd.DataFrame({'feature': feats, 'importance': [scores.get(f, 0) for f in feats]}).sort_values('importance', ascending=False)
    xgb_importance[line] = imp
    print(f'  Top 5: {imp.head(5)["feature"].tolist()}')
    xgb_test_preds[line] = np.maximum(model.predict(dtest), 0.01)
    print(f'  Test pred mean: {xgb_test_preds[line].mean():.4f}')
print(f'\nA v E XGBoost v7 avg D2: {np.mean(list(xgb_val_d2.values())):.4f}')

In [0]:
# Step 3: Two-part model (frequency + severity)
# Frequency: P(claim > 0) using binary XGBoost
# Severity: E[claim | claim > 0] using Gamma XGBoost on positive claims
# Final: freq_pred * severity_pred

tp_val_preds = {}
tp_test_preds = {}
tp_val_d2 = {}

FREQ_PARAMS = {
    'objective': 'binary:logistic',
    'max_depth': 5, 'eta': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.5,
    'min_child_weight': 20, 'reg_alpha': 1.0, 'reg_lambda': 3.0,
    'seed': 42, 'n_jobs': -1, 'verbosity': 0, 'eval_metric': 'logloss',
}
SEV_PARAMS = {
    'objective': 'reg:gamma',  # Gamma for positive continuous
    'max_depth': 5, 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.5,
    'min_child_weight': 10, 'reg_alpha': 0.5, 'reg_lambda': 2.0,
    'seed': 42, 'n_jobs': -1, 'verbosity': 0, 'eval_metric': 'gamma-deviance',
}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}\n=== {line.upper()} Two-Part v7 ===\n{"="*60}')
    train_df = train_v7[line]
    test_df = test[line]
    feats = [f for f in feature_selection[line] if f in train_df.columns and f in test_df.columns]
    
    X_all = train_df[feats].copy()
    X_test = test_df[feats].copy()
    for col in feats:
        if X_all[col].dtype == 'object' or X_all[col].dtype == 'bool':
            cats = pd.Categorical(X_all[col])
            X_all[col] = cats.codes.astype('float64')
            X_test[col] = pd.Categorical(X_test[col], categories=cats.categories).codes.astype('float64')
    
    y_all = train_df['ultimate_claims'].values
    year = train_df['Year'].values
    train_mask = year <= 2023
    val_mask = year == 2024
    
    # Binary target: claim > 0
    y_freq = (y_all > 0).astype(int)
    
    # --- Frequency model (NO offset - binary classification) ---
    dtrain_f = xgb.DMatrix(X_all[train_mask], label=y_freq[train_mask],
                           weight=train_df.loc[train_mask, 'exposure'].values)
    dval_f = xgb.DMatrix(X_all[val_mask], label=y_freq[val_mask],
                         weight=train_df.loc[val_mask, 'exposure'].values)
    dtest_f = xgb.DMatrix(X_test)
    freq_model = xgb.train(FREQ_PARAMS, dtrain_f, num_boost_round=200,
                          evals=[(dval_f, 'val')], early_stopping_rounds=30, verbose_eval=False)
    freq_val = np.clip(freq_model.predict(dval_f), 0.001, 0.999)
    freq_test = np.clip(freq_model.predict(dtest_f), 0.001, 0.999)
    
    # --- Severity model (on positive claims, predict claim AMOUNT directly) ---
    pos_mask = y_all > 0
    pos_train = pos_mask & train_mask
    
    if pos_train.sum() > 50:
        # Split positive training claims into train/val for early stopping
        pos_idx = np.where(pos_train)[0]
        n_pos = len(pos_idx)
        np.random.seed(42)
        np.random.shuffle(pos_idx)
        n_val_pos = max(int(n_pos * 0.15), 10)
        val_pos_idx = pos_idx[:n_val_pos]
        train_pos_idx = pos_idx[n_val_pos:]
        
        y_sev_train = np.maximum(y_all[train_pos_idx], 1.0)  # Gamma requires positive
        y_sev_val = np.maximum(y_all[val_pos_idx], 1.0)
        
        dtrain_s = xgb.DMatrix(X_all.iloc[train_pos_idx], label=y_sev_train,
                               weight=train_df.iloc[train_pos_idx]['exposure'].values)
        dval_s = xgb.DMatrix(X_all.iloc[val_pos_idx], label=y_sev_val,
                            weight=train_df.iloc[val_pos_idx]['exposure'].values)
        dval_all = xgb.DMatrix(X_all[val_mask])
        dtest_s = xgb.DMatrix(X_test)
        
        sev_model = xgb.train(SEV_PARAMS, dtrain_s, num_boost_round=200,
                              evals=[(dval_s, 'val_sev')],
                              early_stopping_rounds=30, verbose_eval=False)
        sev_val = np.maximum(sev_model.predict(dval_all), 1.0)
        sev_test = np.maximum(sev_model.predict(dtest_s), 1.0)
    else:
        print(f'  Not enough positive claims for severity model ({pos_train.sum()})')
        sev_val = np.ones(val_mask.sum()) * y_all[pos_train].mean()
        sev_test = np.ones(len(test_df)) * y_all[pos_train].mean()
    
    # Combine: prediction = freq * severity (pure premium per policy)
    tp_val = freq_val * sev_val
    tp_test = freq_test * sev_test
    
    d2 = tweedie_d2(y_all[val_mask], tp_val)
    tp_val_d2[line] = d2
    tp_val_preds[line] = tp_val
    tp_test_preds[line] = np.maximum(tp_test, 0.01)
    
    print(f'  Freq (val): mean={freq_val.mean():.4f}, actual={y_freq[val_mask].mean():.4f}')
    print(f'  Sev (val): mean={sev_val.mean():.4f}, actual (pos only)={y_all[pos_mask & val_mask].mean():.4f}')
    print(f'  TP Val D2: {d2:.4f}')
    print(f'  TP Val pred mean: {tp_val.mean():.4f} (actual: {y_all[val_mask].mean():.4f})')
    print(f'  TP Test pred mean: {tp_test_preds[line].mean():.4f}')
print(f'\nTwo-Part v7 avg D2: {np.mean(list(tp_val_d2.values())):.4f}')

In [0]:
# Step 3b: GLM v7 (Tweedie with A v E offset, Year + written_premium removed)
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

SMOOTHING = 30
MAX_FEATS = 25

def target_encode(train_series, train_target, apply_series, smoothing=30):
    prior = train_target.mean()
    grp = pd.DataFrame({'cat': train_series, 'target': train_target})
    stats = grp.groupby('cat')['target'].agg(['mean', 'count'])
    smooth = (stats['mean'] * stats['count'] + prior * smoothing) / (stats['count'] + smoothing)
    return train_series.map(smooth).fillna(prior), apply_series.map(smooth).fillna(prior)

glm_val_d2 = {}
glm_val_preds = {}
glm_test_preds = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}\n=== {line.upper()} GLM v7 (A v E offset) ===\n{"="*60}')
    train_df = train_v7[line]
    test_df = test[line]
    top_feats = xgb_importance[line]['feature'].head(MAX_FEATS).tolist()
    # Exclude Year and written_premium from GLM features too
    top_feats = [f for f in top_feats if f in train_df.columns and f in test_df.columns
                 and f not in EXCLUDE_FROM_FEATS]
    print(f'  Features: {len(top_feats)}')
    year = train_df['Year'].values
    train_mask = year <= 2023
    val_mask = year == 2024
    y_tr = train_df.loc[train_mask, 'ultimate_claims'].values
    y_val = train_df.loc[val_mask, 'ultimate_claims'].values
    # A v E offset
    offset_tr = np.log(train_df.loc[train_mask, 'expected_claim'].values + 1e-10)
    offset_val = np.log(train_df.loc[val_mask, 'expected_claim'].values + 1e-10)
    offset_test = np.log(test_df['expected_claim'].values + 1e-10)
    
    X_tr = pd.DataFrame(index=train_df.index[train_mask])
    X_val = pd.DataFrame(index=train_df.index[val_mask])
    X_test_glm = pd.DataFrame(index=test_df.index)
    for col in top_feats:
        tr_vals = train_df.loc[train_mask, col]
        val_vals = train_df.loc[val_mask, col]
        test_vals = test_df[col]
        is_cat = tr_vals.dtype == 'object' or tr_vals.dtype == 'bool'
        if is_cat:
            tr_enc, val_enc = target_encode(tr_vals, y_tr, val_vals, SMOOTHING)
            _, test_enc = target_encode(tr_vals, y_tr, test_vals, SMOOTHING)
            X_tr[col] = tr_enc.values
            X_val[col] = val_enc.values
            X_test_glm[col] = test_enc.values
        else:
            med = tr_vals.median()
            X_tr[col] = tr_vals.fillna(med).values
            X_val[col] = val_vals.fillna(med).values
            X_test_glm[col] = test_vals.fillna(med).values
    scaler = StandardScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
    X_test_s = pd.DataFrame(scaler.transform(X_test_glm), columns=X_test_glm.columns, index=X_test_glm.index)
    X_tr_s = sm.add_constant(X_tr_s, has_constant='add')
    X_val_s = sm.add_constant(X_val_s, has_constant='add')
    X_test_s = sm.add_constant(X_test_s, has_constant='add')
    try:
        model = sm.GLM(y_tr, X_tr_s, family=sm.families.Tweedie(link=sm.families.links.Log(), var_power=1.5),
                       offset=offset_tr)
        result = model.fit(max_iter=100)
        print(f'  Converged: {result.converged}')
        val_pred = np.maximum(result.predict(X_val_s, offset=offset_val), 0)
        d2 = tweedie_d2(y_val, val_pred)
        glm_val_d2[line] = d2
        glm_val_preds[line] = val_pred
        print(f'  Val D2: {d2:.4f}')
        print(f'  Val pred mean: {val_pred.mean():.4f} (actual: {y_val.mean():.4f})')
        glm_test_preds[line] = np.maximum(result.predict(X_test_s, offset=offset_test), 0.01)
        print(f'  Test pred mean: {glm_test_preds[line].mean():.4f}')
    except Exception as e:
        print(f'  GLM FAILED: {e}')
        glm_val_d2[line] = -0.01
        glm_val_preds[line] = np.full(len(y_val), y_tr.mean())
        glm_test_preds[line] = np.full(len(test_df), y_tr.mean())
print(f'\nGLM v7 avg D2: {np.mean(list(glm_val_d2.values())):.4f}')

In [0]:
# Step 4: Blend (Tweedie + Two-Part + GLM) + trend + calibration + submission
print('=== BLEND v7 (Tweedie + Two-Part + GLM) ===')
best_weights = {}
blend_d2 = {}
for line in ['motor', 'buildings', 'contents']:
    train_df = train_v7[line]
    year = train_df['Year'].values
    val_mask = year == 2024
    y_val = train_df.loc[val_mask, 'ultimate_claims'].values
    best_d2 = -999
    best_w = (0.33, 0.33, 0.34)
    # Grid search over 3-way blend
    for wx in np.arange(0, 1.01, 0.1):
        for wt in np.arange(0, 1.01 - wx, 0.1):
            wg = 1.0 - wx - wt
            if wg < 0: continue
            blend = wx * xgb_val_preds[line] + wt * tp_val_preds[line] + wg * glm_val_preds[line]
            d2 = tweedie_d2(y_val, blend)
            if d2 > best_d2:
                best_d2 = d2
                best_w = (wx, wt, wg)
    blend_d2[line] = best_d2
    best_weights[line] = best_w
    print(f'{line}: Tweedie={best_w[0]:.2f}, TwoPart={best_w[1]:.2f}, GLM={best_w[2]:.2f} -> D2={best_d2:.4f}')
print(f'\nBlend v7 avg D2: {np.mean(list(blend_d2.values())):.4f}')
print(f'(v6 blend avg D2 was: 0.0835)')

# Generate calibrated submission
submission = sample_submission.copy()
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    wx, wt, wg = best_weights[line]
    xgb_p = np.clip(xgb_test_preds[line], 0.01, 50000) * NEW_TREND[line]
    tp_p = np.clip(tp_test_preds[line], 0.01, 50000) * NEW_TREND[line]
    glm_p = np.clip(np.nan_to_num(glm_test_preds[line], posinf=50000, neginf=0.01), 0.01, 50000) * NEW_TREND[line]
    blend_pred = np.clip(wx * xgb_p + wt * tp_p + wg * glm_p, 0.01, 50000)
    cal_target = train_v7[line].loc[train_v7[line]['Year'] == 2024, 'ultimate_claims'].mean() * NEW_TREND[line]
    cal_factor = cal_target / max(blend_pred.mean(), 0.01)
    blend_pred = np.clip(blend_pred * cal_factor, 0.01, 50000)
    print(f'{line}: blend mean={blend_pred.mean():.4f} (target={cal_target:.4f})')
    pred_map = dict(zip(test_df['row_id'], blend_pred))
    mask = submission['row_id'].isin(pred_map)
    submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
submission['prediction'] = submission['prediction'].clip(lower=0.01, upper=50000)
submission['prediction'] = submission['prediction'].replace([np.inf, -np.inf], 50000).fillna(0.01)
sub_path = os.path.join(SUB_DIR, 'blend_v7.csv')
submission.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} ({len(submission)} rows, mean={submission["prediction"].mean():.4f})')

# Save results
import pickle
v7_results = {'xgb_val_d2': xgb_val_d2, 'tp_val_d2': tp_val_d2, 'blend_d2': blend_d2,
              'best_weights': best_weights, 'trend_factors': NEW_TREND, 'lr_2021': LR_2021}
with open(os.path.join(V7_DIR, 'v7_results.pkl'), 'wb') as f:
    pickle.dump(v7_results, f)
print(f'Saved: {V7_DIR}v7_results.pkl')
print('\n=== v7 Developed Market Complete ===')